# Phase 5 — Embeddings & Vector Search with FAISS

**Goal:** Convert document text into numeric vectors so we can find the most relevant passage for any question. (`src/store/faiss_store.py`)

---

## Why do we need this?

After OCR, cleaning, and NER we have raw text and entity metadata.  
When a user asks a question we need to find the **most relevant chunk of text** to answer it.  
Keyword search (e.g. `if word in text`) fails when the user phrases things differently from the document.

**Semantic search** solves this — it compares *meaning*, not exact words:

```
Question : "How much did the software work cost?"
Document : "Software Development  Rs 10,000"       ← no word overlap, but same meaning
```

To compare meaning we turn every piece of text into a **vector** (a list of numbers).  
Vectors that are close together in space have similar meanings.

---

## The full flow

```
Document text
      │
      ▼
┌─────────────────┐
│  chunk_text()   │  Split into overlapping windows so no sentence is cut in half
└────────┬────────┘
         │  list of text chunks
         ▼
┌─────────────────┐
│ SentenceTransf. │  Encode each chunk → 384-dimensional float vector
└────────┬────────┘
         │  numpy array  shape (n_chunks, 384)
         ▼
┌─────────────────┐
│  FAISS index    │  Store all vectors; supports fast nearest-neighbour lookup
└────────┬────────┘
         │
  At query time:
         │
  User question  ──► encode ──► query vector
         │
         ▼
  FAISS returns the k closest vectors  →  retrieve matching text chunks
         │
         ▼
  Chunks passed to Claude as context  →  answer generated
```

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────
# sentence-transformers: pre-trained models that turn text into dense vectors
from sentence_transformers import SentenceTransformer

# faiss: Facebook AI Similarity Search — fast nearest-neighbour search over vectors
import faiss

# numpy: FAISS works with numpy arrays, not plain Python lists
import numpy as np

# json + os: used later to save/load the store to disk
import json
import os

## What are sentence embeddings?

A **sentence embedding** is a fixed-length vector that encodes the *meaning* of a piece of text.

The model is trained on millions of sentence pairs so that sentences with similar meanings end up close together in vector space:

```
"The cat sat on the mat"   →  [0.12, -0.45, 0.87, ...]  ← 384 numbers
"A kitten rested on a rug" →  [0.11, -0.43, 0.85, ...]  ← very close — same meaning!
"The stock market crashed" →  [-0.9,  0.23, -0.1, ...]  ← far away   — different topic
```

### Why `all-MiniLM-L6-v2`?

| Model | Dimensions | Size | Speed |
|-------|-----------|------|-------|
| `all-MiniLM-L6-v2` | 384 | ~80 MB | Fast — good default |
| `all-mpnet-base-v2` | 768 | ~420 MB | Slower, more accurate |

`all-MiniLM-L6-v2` is the standard starting point — small, fast, and accurate enough for document Q&A.

### Cosine similarity

We measure closeness using **cosine similarity** — the angle between two vectors.  
A score of `1.0` means identical meaning; `0.0` means unrelated; `-1.0` means opposite.

```
cosine_similarity(a, b) = (a · b) / (|a| × |b|)
```

When vectors are **L2-normalised** (forced to length = 1), cosine similarity equals the dot product.  
FAISS's `IndexFlatIP` computes dot products efficiently — so normalise first, then use `IndexFlatIP`.

In [ ]:
# ── Load the embedding model ───────────────────────────────────────────
# First run downloads ~80 MB from HuggingFace and caches it locally.
# Subsequent runs load from the local cache — no internet needed.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Check the output dimension — this is the length of every vector the model produces
print(f'Embedding dimension: {model.get_sentence_embedding_dimension()}')
# Expected: 384

# ── Encode a few example sentences ────────────────────────────────────
# encode() accepts a list of strings and returns a 2-D numpy array
# shape: (number_of_sentences, embedding_dimension) → (3, 384) here
sentences = [
    "How much did the software work cost?",   # a question
    "Software Development Rs 10,000",          # the matching invoice line
    "The cat sat on the mat",                  # unrelated sentence
]

embeddings = model.encode(sentences)
print(f'Embeddings shape     : {embeddings.shape}')       # (3, 384)
print(f'First vector (5 vals): {embeddings[0][:5]}')

In [ ]:
# ── Manually compute cosine similarity to build intuition ──────────────
# This is what FAISS does internally after normalisation.
# We do it manually here so you can see the numbers and understand the concept.

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    # Divide each vector by its magnitude to get a unit vector (length = 1)
    a_norm = a / np.linalg.norm(a)
    b_norm = b / np.linalg.norm(b)
    # Dot product of two unit vectors equals their cosine similarity
    return float(np.dot(a_norm, b_norm))

# Compare the question against the invoice line and the unrelated sentence
sim_question_invoice = cosine_sim(embeddings[0], embeddings[1])
sim_question_cat     = cosine_sim(embeddings[0], embeddings[2])

print(f'Question ↔ Invoice line  : {sim_question_invoice:.3f}')  # should be high
print(f'Question ↔ Cat sentence  : {sim_question_cat:.3f}')      # should be low

## What is FAISS?

**FAISS** (Facebook AI Similarity Search) is a library for efficiently searching among large collections of vectors.

Naively finding the closest vector requires comparing your query against *every* stored vector — O(n).  
FAISS uses optimised data structures and CPU instructions to make this much faster.

### Index types

| Index | How it works | Use case |
|-------|-------------|----------|
| `IndexFlatL2` | Exact search — Euclidean distance | Small datasets, exact results |
| `IndexFlatIP` | Exact search — inner product (dot product) | **Our choice** — cosine sim after normalisation |
| `IndexIVFFlat` | Approximate search — partitions space into clusters | Large datasets (>100k vectors) |

We use **`IndexFlatIP`** because:
1. Our document collections are small (hundreds of chunks, not millions) — exact search is fine
2. After L2-normalising the vectors, inner product equals cosine similarity
3. No training step required — `IndexFlatIP` is ready to use immediately

### What FAISS does NOT store

FAISS only stores **float vectors and their integer IDs**.  
It has no concept of text — we maintain a separate `chunks` list that maps each integer ID back to the original text.  
This is why the chunks list and the FAISS index must always stay in sync.

In [ ]:
# ── Create a FAISS index ───────────────────────────────────────────────
# IndexFlatIP performs exact inner-product search
# We pass the embedding dimension so FAISS knows how large each vector is
dimension = model.get_sentence_embedding_dimension()   # 384
index = faiss.IndexFlatIP(dimension)

print(f'Index created. Vectors stored: {index.ntotal}')   # 0 — empty so far

# ── Normalise and add the example embeddings ───────────────────────────
# FAISS requires float32; sentence-transformers returns float32 by default
vecs = embeddings.astype('float32')

# faiss.normalize_L2 divides each row vector by its own magnitude IN PLACE
# After this step: inner product between any two vectors == cosine similarity
faiss.normalize_L2(vecs)

# Add all vectors — FAISS assigns IDs 0, 1, 2 automatically
index.add(vecs)
print(f'Vectors stored after add: {index.ntotal}')   # 3

# ── Run a search ───────────────────────────────────────────────────────
# Encode the query, normalise it (same steps as the document vectors)
query = model.encode(["What is the cost of software development?"]).astype('float32')
faiss.normalize_L2(query)

# search() returns two arrays of shape (1, k):
#   scores — similarity scores, higher is more relevant
#   ids    — index positions of the matching vectors
scores, ids = index.search(query, k=2)   # k=2: return the 2 closest results

print()
print('Top results:')
for rank, (score, idx) in enumerate(zip(scores[0], ids[0])):
    print(f'  Rank {rank+1}: score={score:.3f}  →  "{sentences[idx]}"')

## Chunking — why we split documents

We cannot encode an entire document as a single vector for two reasons:

1. **Token limit** — sentence-transformer models have a max input length (~256–512 tokens). Text beyond that is silently truncated, losing information.
2. **Meaning dilution** — a vector for a 3-page document averages out all topics. A question about invoice totals would score low against a document where totals appear on just one line.

The solution: **chunk** the text into smaller overlapping windows before encoding.

### Why overlap?

If we split on hard word-count boundaries a sentence that spans a boundary gets cut, losing context.  
**Overlap** means adjacent chunks share words so no sentence is orphaned:

```
Text:  word1 word2 word3 word4 word5 word6 word7 word8

chunk_size=4, overlap=1:
  Chunk 0:  word1 word2 word3 word4
  Chunk 1:  word4 word5 word6 word7   ← word4 repeated so context carries over
  Chunk 2:  word7 word8
```

In [ ]:
def chunk_text(text: str, chunk_size: int = 100, overlap: int = 20) -> list[str]:
    """
    Split text into overlapping windows of words.

    Args:
        text       : raw document string
        chunk_size : number of words per chunk
        overlap    : number of words shared between consecutive chunks

    Returns:
        list of text chunk strings
    """
    # Split on whitespace — gives a flat list of every word in the document
    words = text.split()

    chunks = []

    # Move the window forward by (chunk_size - overlap) each step
    # so the overlap region is included in both the current and next chunk
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        end = start + chunk_size          # end of this window
        chunk = words[start:end]          # Python slicing won't go past list end
        chunks.append(' '.join(chunk))    # rejoin words back into a readable string

    return chunks


# ── Demo: see the overlapping windows ─────────────────────────────────
demo_text = "word1 word2 word3 word4 word5 word6 word7 word8 word9 word10"
for i, chunk in enumerate(chunk_text(demo_text, chunk_size=4, overlap=1)):
    print(f'Chunk {i}: {chunk}')

## Indexing a full document

Now we combine chunking and FAISS on a real document.

Steps:
1. **Chunk** the text into overlapping windows
2. **Encode** all chunks in one batch (batch processing is faster than one-by-one)
3. **Normalise** the vectors so inner product equals cosine similarity
4. **Add** to a fresh FAISS index — IDs are assigned automatically as 0, 1, 2 ...
5. **Mirror** the chunks in a plain list — this is our ID → text lookup table

In [ ]:
# ── Sample invoice document (from notebook 01 OCR output) ─────────────
invoice_text = """INVOICE
Invoice No: INV-2024-0451
Date: March 15, 2024
Bill To: GlobalTech Ltd
Address: Mumbai, India
Description          Amount
Software Development Rs 10,000
Consulting Services  Rs 2,450
TOTAL: Rs 12,450
GST @ 18%: Rs 2,241
"""

# ── Step 1: Chunk the document ─────────────────────────────────────────
# chunk_size=30, overlap=5 works well for short structured documents like invoices
# For long free-text documents (CVs, contracts) use chunk_size=100-150
chunks = chunk_text(invoice_text, chunk_size=30, overlap=5)
print(f'Document split into {len(chunks)} chunk(s)')
for i, c in enumerate(chunks):
    print(f'  [{i}] {c}')

# ── Step 2: Batch encode all chunks ───────────────────────────────────
# Batch encoding is faster because the model processes them in parallel internally
# Result shape: (n_chunks, 384)
chunk_vecs = model.encode(chunks, show_progress_bar=False).astype('float32')
print(f'\nEncoded shape: {chunk_vecs.shape}')

# ── Step 3: Create index, normalise, and add ───────────────────────────
doc_index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(chunk_vecs)    # make inner product == cosine similarity
doc_index.add(chunk_vecs)         # IDs auto-assigned: 0, 1, 2, ...

# ── Step 4: Keep a parallel list of chunk texts ────────────────────────
# FAISS returns integer IDs — we use this list to convert ID → original text
chunk_store = chunks

print(f'Index contains {doc_index.ntotal} vectors')

## Searching — how a question becomes a query vector

At query time we follow the **same encode → normalise steps** as we did for the document chunks.  
This is critical — if you normalise chunks but not the query (or vice versa) the similarity scores will be wrong.

`index.search(query, k)` returns:
- `scores` — shape `(1, k)` — similarity score per result; higher = more relevant with `IndexFlatIP`
- `ids` — shape `(1, k)` — position in the index, used to look up text in `chunk_store`

The outer dimension of both arrays is the **number of queries** — since we search one question at a time it's always `1`, so we always index with `[0]`.

The retrieved text chunks are then passed to Claude as **context** so it can construct an answer.

In [ ]:
def search_index(question: str, index: faiss.Index, chunk_store: list[str], k: int = 3) -> list[dict]:
    """
    Find the k most relevant text chunks for a given question.

    Args:
        question    : user's natural-language question
        index       : the populated FAISS index
        chunk_store : list of text chunks; position i matches FAISS vector ID i
        k           : number of results to return

    Returns:
        list of dicts with keys 'text' and 'score', ordered by relevance (best first)
    """
    # Encode the question using the same model used to encode the chunks
    # shape: (1, 384) — a single vector wrapped in a 2-D array
    query_vec = model.encode([question]).astype('float32')

    # Normalise the query vector — must match the normalisation applied to chunks
    faiss.normalize_L2(query_vec)

    # FAISS search — returns arrays of shape (1, k)
    scores, ids = index.search(query_vec, k=k)

    results = []
    # ids[0] and scores[0] because the first dimension is the number of queries (1)
    for score, idx in zip(scores[0], ids[0]):
        # FAISS fills unused slots with -1 when fewer than k results exist
        if idx == -1:
            continue
        results.append({
            "text":  chunk_store[idx],
            "score": round(float(score), 3),
        })
    return results


# ── Test with questions about the invoice ─────────────────────────────
questions = [
    "What is the total amount on the invoice?",
    "Who is the invoice billed to?",
    "What is the GST amount?",
]

for q in questions:
    print(f'Q: {q}')
    for r in search_index(q, doc_index, chunk_store, k=2):
        print(f'  [score={r["score"]}]  {r["text"]}')
    print()

## The `FaissStore` class — what `src/store/faiss_store.py` will implement

In the actual pipeline we need more than a one-shot index:
- **Add** multiple documents over time, not just one
- **Store metadata** per chunk so the agent knows the source and document type
- **Save** the index to disk so we don't re-encode on every restart
- **Load** a saved index back at startup

The `FaissStore` class wraps all of this:

```
FaissStore
  ├── index       : faiss.IndexFlatIP   — the vector index
  ├── chunks      : list[str]           — text for each chunk (position = vector ID)
  ├── metadata    : list[dict]          — doc_type, source path, etc. per chunk
  ├── add()       — chunk → encode → normalise → add to index
  ├── search()    — encode question → normalise → return top-k chunks with metadata
  ├── save()      — write index.faiss + chunks.json to disk
  └── load()      — restore everything from disk (classmethod)
```

### Why store metadata alongside chunks?

Each chunk should know which document it came from and what type that document was.  
This lets the agent filter results (e.g. "only search invoice chunks") and cite sources in answers.  
Metadata is stored as a plain dict per chunk — e.g. `{"doc_type": "invoice", "source": "inv_001.png"}`.

In [ ]:
class FaissStore:
    """
    Wrapper around a FAISS IndexFlatIP that keeps text chunks and
    per-chunk metadata in sync with the vector index.
    """

    def __init__(self, embedding_model: SentenceTransformer):
        # Keep a reference to the model — every add() and search() call uses the
        # same model to ensure query and chunk vectors live in the same embedding space
        self.model = embedding_model
        dim = embedding_model.get_sentence_embedding_dimension()

        # Create an empty exact inner-product FAISS index
        # After L2-normalisation, inner product == cosine similarity
        self.index = faiss.IndexFlatIP(dim)

        # These two lists are always the same length as index.ntotal
        # Position i in each list corresponds to vector ID i in FAISS
        self.chunks   = []   # raw text of each chunk
        self.metadata = []   # dict of extra info per chunk (doc_type, source, etc.)

    def add(self, text: str, metadata: dict = None) -> None:
        """
        Chunk a document, encode each chunk, and add to the index.

        Args:
            text     : raw document text
            metadata : dict to attach to every chunk from this document
                       e.g. {'doc_type': 'invoice', 'source': 'inv_001.png'}
        """
        if metadata is None:
            metadata = {}

        # Split the document into overlapping word windows
        new_chunks = chunk_text(text)

        # Batch-encode all chunks at once — the model processes them in parallel
        # which is significantly faster than calling encode() in a loop
        vecs = self.model.encode(new_chunks).astype('float32')

        # Normalise so inner product == cosine similarity when searching
        faiss.normalize_L2(vecs)

        # Add to the FAISS index — IDs continue from wherever ntotal left off
        self.index.add(vecs)

        # Append the new chunk texts to keep the lookup list in sync with the index
        self.chunks.extend(new_chunks)

        # Each chunk from this document gets the same metadata dict
        self.metadata.extend([metadata] * len(new_chunks))

    def search(self, question: str, k: int = 3) -> list[dict]:
        """
        Find the k most relevant chunks for a question.

        Args:
            question : user's natural-language question
            k        : number of top results to return

        Returns:
            list of dicts: {'text', 'score', 'metadata'}, ordered best-first
        """
        # Encode and normalise the question exactly as we did for document chunks
        # Consistency here is critical — same model, same normalisation
        query_vec = self.model.encode([question]).astype('float32')
        faiss.normalize_L2(query_vec)

        # FAISS returns (scores, ids) each of shape (1, k)
        scores, ids = self.index.search(query_vec, k=k)

        results = []
        for score, idx in zip(scores[0], ids[0]):
            # FAISS fills empty result slots with -1 when the index has fewer than k vectors
            if idx == -1:
                continue
            results.append({
                "text":     self.chunks[idx],
                "score":    round(float(score), 3),
                "metadata": self.metadata[idx],
            })
        return results

    def save(self, directory: str) -> None:
        """
        Persist the index and chunk data to disk.

        Writes:
            <directory>/index.faiss  — binary FAISS index file
            <directory>/chunks.json  — chunk texts and metadata as JSON
        """
        # Create the target directory if it doesn't exist
        os.makedirs(directory, exist_ok=True)

        # faiss.write_index serialises the index to a binary file
        faiss.write_index(self.index, os.path.join(directory, 'index.faiss'))

        # Save chunks + metadata together in one file so they can't get out of sync
        with open(os.path.join(directory, 'chunks.json'), 'w') as f:
            json.dump({'chunks': self.chunks, 'metadata': self.metadata}, f)

        print(f'Saved {self.index.ntotal} vectors to "{directory}"')

    @classmethod
    def load(cls, directory: str, embedding_model: SentenceTransformer) -> 'FaissStore':
        """
        Restore a previously saved FaissStore from disk.

        Args:
            directory       : folder containing index.faiss and chunks.json
            embedding_model : must be the SAME model used when the store was built
                              — a different model would produce incompatible vectors

        Returns:
            A fully restored FaissStore instance
        """
        # Create a blank store, then overwrite its index and lists with saved data
        store = cls(embedding_model)

        # faiss.read_index deserialises the binary file back into a FAISS Index object
        store.index = faiss.read_index(os.path.join(directory, 'index.faiss'))

        # Restore the chunk texts and metadata lists
        with open(os.path.join(directory, 'chunks.json'), 'r') as f:
            data = json.load(f)
        store.chunks   = data['chunks']
        store.metadata = data['metadata']

        print(f'Loaded {store.index.ntotal} vectors from "{directory}"')
        return store

## End-to-end demo — add, search, save, and reload

This is the exact flow the Q&A agent will use:

1. **Ingest time** — call `store.add()` for each new document
2. **Query time** — call `store.search()` to retrieve the most relevant chunks
3. **Shutdown** — call `store.save()` so the index survives a restart
4. **Startup** — call `FaissStore.load()` instead of re-encoding every document from scratch

In [ ]:
# ── Build and populate the store with two different documents ──────────
store = FaissStore(model)

# Add the invoice document — attach metadata so search results are traceable
store.add(invoice_text, metadata={"doc_type": "invoice", "source": "test_invoice.png"})

# Add a CV document so we can test cross-document search
cv_text = """Kevin William — Data Scientist / Machine Learning Engineer
Experience: Information Tech Consultants, Data Scientist, March 2026 - Present.
Sika, Data Scientist / Quality Engineer, October 2022 - October 2025.
Education: The University of Manchester, MChem Chemistry, 2017-2021.
Skills: Python, TensorFlow, scikit-learn, Pandas, NLP, machine learning."""

store.add(cv_text, metadata={"doc_type": "cv", "source": "Profile.pdf"})

print(f'Total vectors in store: {store.index.ntotal}')

# ── Search across both documents ───────────────────────────────────────
# FAISS searches all vectors regardless of document — the closest win
# The metadata lets us see which document each result came from
print()
for question in ["What is the invoice total?", "Where did Kevin study?"]:
    print(f'Q: {question}')
    for r in store.search(question, k=2):
        src = r['metadata'].get('source', '?')
        print(f'  [score={r["score"]} | {src}]  {r["text"]}')
    print()

# ── Save the store to disk ─────────────────────────────────────────────
# This writes index.faiss and chunks.json into the data/faiss_store folder
STORE_DIR = '../data/faiss_store'
store.save(STORE_DIR)

# ── Reload the store from disk and verify it still works ──────────────
# load() is a classmethod so we call it on the class, not an instance
loaded_store = FaissStore.load(STORE_DIR, model)

# Run a search on the reloaded store to confirm it matches the original
print()
print('Search on reloaded store:')
for r in loaded_store.search("What is the total amount?", k=2):
    print(f'  [score={r["score"]} | {r["metadata"]["source"]}]  {r["text"]}')